In [42]:
import pandas as pd

df_results = pd.read_csv("data/results.csv")
df_goalscorers = pd.read_csv("data/goalscorers.csv")
df_shootouts = pd.read_csv("data/shootouts.csv")
df_former_names = pd.read_csv("data/former_names.csv")

print(f"Loaded {len(df_results):,} results, {len(df_goalscorers):,} goalscorers, {len(df_shootouts):,} shootouts")
df_results.head()

Loaded 49,477 results, 47,690 goalscorers, 678 shootouts


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [43]:
print("filtering datasets...")

filtering datasets...


In [44]:
competitive_tournaments = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Copa Am\u00e9rica',
    'Copa Am\u00e9rica qualification',
]

In [45]:
df_filtered = df_results[df_results['tournament'].isin(competitive_tournaments)].copy()
df_filtered = df_filtered.sort_values(by='date').reset_index(drop=True)

print(f"Filtering complete! Kept {len(df_filtered):,} competitive matches out of {len(df_results):,} total.")
df_filtered.head()

Filtering complete! Kept 13,896 competitive matches out of 49,477 total.


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1916-07-02,Chile,Uruguay,0.0,4.0,Copa América,Buenos Aires,Argentina,True
1,1916-07-06,Argentina,Chile,6.0,1.0,Copa América,Buenos Aires,Argentina,False
2,1916-07-08,Brazil,Chile,1.0,1.0,Copa América,Buenos Aires,Argentina,True
3,1916-07-10,Argentina,Brazil,1.0,1.0,Copa América,Buenos Aires,Argentina,False
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Copa América,Buenos Aires,Argentina,True


In [46]:
print("Creating target variables...")

#Full time winners only
def get_regular_winners(row):
    if row['home_score'] > row['away_score']:
        return row['home_team']
    elif row['away_score'] > row['home_score']:
        return row['away_team']
    else:
        return 'Draw'


df_filtered['winner'] = df_filtered.apply(get_regular_winners, axis=1)

# 2. Merge shootout winners for matches that ended as a Draw
# We match them up by date, home_team, and away_team
df_merged = pd.merge(
    df_filtered, 
    df_shootouts[['date', 'home_team', 'away_team', 'winner']], 
    on=['date', 'home_team', 'away_team'], 
    how='left', 
    suffixes=('', '_shootout')
)

# 3. If there was a shootout winner, override the 'Draw' status
df_merged['winner'] = df_merged['winner_shootout'].fillna(df_merged['winner'])

# 4. Clean up the temporary shootout column
df_clean = df_merged.drop(columns=['winner_shootout'])

print("Winner column successfully created and adjusted for penalty shootouts!")
df_clean[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'winner']].head(10)

Creating target variables...
Winner column successfully created and adjusted for penalty shootouts!


,date,home_team,away_team,home_score,away_score,winner
0,1916-07-02,Chile,Uruguay,0.0,4.0,Uruguay
1,1916-07-06,Argentina,Chile,6.0,1.0,Argentina
2,1916-07-08,Brazil,Chile,1.0,1.0,Draw
3,1916-07-10,Argentina,Brazil,1.0,1.0,Draw
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Uruguay
5,1916-07-17,Argentina,Uruguay,0.0,0.0,Draw
6,1917-09-30,Uruguay,Chile,4.0,0.0,Uruguay
7,1917-10-03,Argentina,Brazil,4.0,2.0,Argentina
8,1917-10-06,Argentina,Chile,1.0,0.0,Argentina
9,1917-10-07,Uruguay,Brazil,4.0,0.0,Uruguay


In [47]:
# Cell 4: Calculating Rolling Recent Form
print("Calculating team form metrics...")

# 1. Create a helper function to determine points earned by home and away teams
def calculate_points(row):
    if row['winner'] == 'Draw':
        return 1, 1
    elif row['winner'] == row['home_team']:
        return 3, 0
    else:
        return 0, 3

# Apply the function to get points for every match
points = df_clean.apply(calculate_points, axis=1)
df_clean['home_points'] = [p[0] for p in points]
df_clean['away_points'] = [p[1] for p in points]

# 2. Re-organize data to calculate a timeline of points for every single country
team_matches = []
for index, row in df_clean.iterrows():
    team_matches.append({'date': row['date'], 'team': row['home_team'], 'points': row['home_points']})
    team_matches.append({'date': row['date'], 'team': row['away_team'], 'points': row['away_points']})

df_teams = pd.DataFrame(team_matches).sort_values(by=['team', 'date'])

# 3. Calculate the rolling average points over the last 5 matches (excluding the current match)
df_teams['form_last_5'] = df_teams.groupby('team')['points'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# 4. Map these form metrics back into our main match dataset
df_clean = pd.merge(df_clean, df_teams[['date', 'team', 'form_last_5']], 
                    left_on=['date', 'home_team'], right_on=['date', 'team'], how='left').rename(columns={'form_last_5': 'home_form'})
df_clean.drop(columns=['team'], inplace=True)

df_clean = pd.merge(df_clean, df_teams[['date', 'team', 'form_last_5']], 
                    left_on=['date', 'away_team'], right_on=['date', 'team'], how='left').rename(columns={'form_last_5': 'away_form'})
df_clean.drop(columns=['team'], inplace=True)

# Fill initial matches (where a team hasn't played 5 games yet) with the average baseline
df_clean['home_form'] = df_clean['home_form'].fillna(1.0)
df_clean['away_form'] = df_clean['away_form'].fillna(1.0)

print("Team form metrics successfully engineered!")
df_clean[['date', 'home_team', 'away_team', 'home_form', 'away_form', 'winner']].tail(10)

Calculating team form metrics...
Team form metrics successfully engineered!


,date,home_team,away_team,home_form,away_form,winner
13886,2026-06-26,Norway,France,3.0,2.6,Draw
13887,2026-06-26,Egypt,Iran,2.2,1.2,Draw
13888,2026-06-26,New Zealand,Belgium,2.0,1.8,Draw
13889,2026-06-26,Senegal,Iraq,1.8,1.4,Draw
13890,2026-06-27,DR Congo,Uzbekistan,2.0,1.0,Draw
13891,2026-06-27,Colombia,Portugal,2.6,1.6,Draw
13892,2026-06-27,Panama,England,1.4,2.6,Draw
13893,2026-06-27,Algeria,Austria,2.0,1.4,Draw
13894,2026-06-27,Jordan,Argentina,0.8,2.0,Draw
13895,2026-06-27,Croatia,Ghana,2.4,2.6,Draw


In [48]:
# Cell 5: Preparing features and encoding the target variable
print("Encoding data for the model...")

# 1. Define our input features (X) and what we want to predict (y)
def encode_target(row):
    if row['winner'] == row['home_team']:
        return 2  # Home Win
    elif row['winner'] == 'Draw':
        return 1  # Draw
    else:
        return 0  # Away Win / Loss

df_clean['target'] = df_clean.apply(encode_target, axis=1)

# 2. Isolate the feature columns and target vector
features = ['home_form', 'away_form', 'home_score', 'away_score'] # Base columns we've built
X = df_clean[['home_form', 'away_form']] # We only feed form metrics to avoid leaking the final score!
y = df_clean['target']

print("Target variables successfully encoded into integers!")
print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")
print("\nFirst 5 rows of our finalized training features (X):")
print(X.head())

Encoding data for the model...
Target variables successfully encoded into integers!
Features shape (X): (13896, 2)
Target shape (y): (13896,)

First 5 rows of our finalized training features (X):
   home_form  away_form
0        1.0        1.0
1        1.0        0.0
2        1.0        0.0
3        3.0        1.0
4        1.0        3.0


In [49]:
# Cell 6: Model Training and Evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Initializing machine learning model training...")

# 1. Split data: 80% to train the model, 20% to test its accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)

# 2. Initialize and train the Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 3. Make predictions on our test dataset to evaluate it
y_pred = model.predict(X_test)

# 4. Calculate overall accuracy score
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Model Training Complete!")
print(f"==============================")
print(f" Overall Baseline Accuracy: {accuracy * 100:.2f}%")
print(f"==============================")
print("\n Detailed Performance Report:")
print(classification_report(y_test, y_pred, target_names=['Away Win', 'Draw', 'Home Win']))

Initializing machine learning model training...

 Model Training Complete!
 Overall Baseline Accuracy: 57.48%

 Detailed Performance Report:
              precision    recall  f1-score   support

    Away Win       0.51      0.53      0.52       823
        Draw       0.00      0.00      0.00       566
    Home Win       0.60      0.84      0.70      1391

    accuracy                           0.57      2780
   macro avg       0.37      0.45      0.41      2780
weighted avg       0.45      0.57      0.50      2780



c:\Users\Mazum Paudel\Desktop\world-cup-predictor\world-cup-predictor\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Mazum Paudel\Desktop\world-cup-predictor\world-cup-predictor\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Mazum Paudel\Desktop\world-cup-predictor\world-cup-predictor\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zer

In [50]:
# Cell 7: Simulating a 2026 World Cup Match Predictor
print("Initializing the Predictor Engine...")

def predict_match(home_team, away_team):
    # 1. Look up the most recent calculated form for both teams from our df_teams data
    try:
        home_f = df_teams[df_teams['team'] == home_team]['form_last_5'].iloc[-1]
    except:
        home_f = 1.0 # Default baseline if team isn't found
        
    try:
        away_f = df_teams[df_teams['team'] == away_team]['form_last_5'].iloc[-1]
    except:
        away_f = 1.0

    # 2. Package the features into a small DataFrame matching our model's input structure
    match_features = pd.DataFrame([[home_f, away_f]], columns=['home_form', 'away_form'])
    
    # 3. Get probability distributions from our model
    # Class order: [Away Win, Draw, Home Win]
    probabilities = model.predict_proba(match_features)[0]
    
    # 4. Print out the formatted predictions
    print(f"\n Match Simulation: {home_team} vs. {away_team}")
    print(f"--------------------------------------------------")
    print(f"{home_team} (Home) Current Form Rating: {home_f:.2f}")
    print(f"{away_team} (Away) Current Form Rating: {away_f:.2f}")
    print(f"--------------------------------------------------")
    print(f"Win Probability for {home_team}: {probabilities[2]*100:.1f}%")
    print(f"Draw Probability: {probabilities[1]*100:.1f}%")
    print(f"Win Probability for {away_team}: {probabilities[0]*100:.1f}%")

print("Predictor Engine Ready!")

Initializing the Predictor Engine...
Predictor Engine Ready!


In [51]:
predict_match("Uruguay", "Spain")


 Match Simulation: Uruguay vs. Spain
--------------------------------------------------
Uruguay (Home) Current Form Rating: 1.80
Spain (Away) Current Form Rating: 2.20
--------------------------------------------------
Win Probability for Uruguay: 41.2%
Draw Probability: 25.8%
Win Probability for Spain: 33.0%


In [52]:
predict_match("Norway", "France")


 Match Simulation: Norway vs. France
--------------------------------------------------
Norway (Home) Current Form Rating: 3.00
France (Away) Current Form Rating: 2.60
--------------------------------------------------
Win Probability for Norway: 55.7%
Draw Probability: 26.1%
Win Probability for France: 18.2%


In [53]:
# Cell 8: Feature Engineering for Goals Scored and Conceded
print("Calculating attacking and defensive metrics...")

# 1. Re-organize data to create a timeline of goals scored and conceded for every country
goal_timeline = []
for index, row in df_clean.iterrows():
    # Home team stats for this match
    goal_timeline.append({
        'date': row['date'], 
        'team': row['home_team'], 
        'goals_scored': row['home_score'], 
        'goals_conceded': row['away_score']
    })
    # Away team stats for this match
    goal_timeline.append({
        'date': row['date'], 
        'team': row['away_team'], 
        'goals_scored': row['away_score'], 
        'goals_conceded': row['home_score']
    })

df_goals = pd.DataFrame(goal_timeline).sort_values(by=['team', 'date'])

# 2. Calculate rolling averages for last 5 matches (excluding the current match using .shift())
df_goals['avg_goals_scored_5'] = df_goals.groupby('team')['goals_scored'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)
df_goals['avg_goals_conceded_5'] = df_goals.groupby('team')['goals_conceded'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# 3. Merge these features back into our main df_clean dataframe
df_clean = pd.merge(df_clean, df_goals[['date', 'team', 'avg_goals_scored_5', 'avg_goals_conceded_5']], 
                    left_on=['date', 'home_team'], right_on=['date', 'team'], how='left').rename(
                    columns={'avg_goals_scored_5': 'home_avg_goals_scored', 'avg_goals_conceded_5': 'home_avg_goals_conceded'})
df_clean.drop(columns=['team'], inplace=True)

df_clean = pd.merge(df_clean, df_goals[['date', 'team', 'avg_goals_scored_5', 'avg_goals_conceded_5']], 
                    left_on=['date', 'away_team'], right_on=['date', 'team'], how='left').rename(
                    columns={'avg_goals_scored_5': 'away_avg_goals_scored', 'avg_goals_conceded_5': 'away_avg_goals_conceded'})
df_clean.drop(columns=['team'], inplace=True)

# 4. Fill initial missing rows with a standard baseline (e.g., 1.0 goal per match)
df_clean.fillna({'home_avg_goals_scored': 1.0, 'home_avg_goals_conceded': 1.0, 
                 'away_avg_goals_scored': 1.0, 'away_avg_goals_conceded': 1.0}, inplace=True)

print("Advanced goal metrics successfully engineered!")
df_clean[['date', 'home_team', 'away_team', 'home_form', 'home_avg_goals_scored', 'home_avg_goals_conceded']].tail(5)

Calculating attacking and defensive metrics...
Advanced goal metrics successfully engineered!


,date,home_team,away_team,home_form,home_avg_goals_scored,home_avg_goals_conceded
13891,2026-06-27,Colombia,Portugal,2.6,2.8,1.0
13892,2026-06-27,Panama,England,1.4,1.4,1.0
13893,2026-06-27,Algeria,Austria,2.0,1.4,1.0
13894,2026-06-27,Jordan,Argentina,0.8,1.2,1.4
13895,2026-06-27,Croatia,Ghana,2.4,2.4,1.4
